In [144]:
import os
import pandas as pd 
from glob import glob
from functools import reduce
from xml.etree import ElementTree as et

In [145]:
# list all xml file store in a list
xml_list = glob('./data_images/*.xml')

In [147]:
#data clean replace \\ with / 
xml_file=list(map(lambda x: x.replace('\\','/'),xml_list))

In [148]:
# step-2 read xml files
# from each xml file we need to extract 
# filename , size(width , height) , object(xmin, xmax, ymin, ymax)
def extract(filename):
    tree = et.parse(filename)
    root = tree.getroot()
    image_name = root.find('filename').text
    size = root.find('size')
    width = size.find('width').text
    height = size.find('height').text
    objs = root.findall('object')
    parser = []
    for obj in objs:
        name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = bndbox.find('xmin').text
        xmax = bndbox.find('xmax').text
        ymin = bndbox.find('ymin').text
        ymax = bndbox.find('ymax').text
        parser.append([image_name,width,height,name,xmin,xmax,ymin,ymax])
    return parser


In [149]:
# all detail of object
parser_all = list(map(extract,xml_file))

In [150]:
# reducing the the data
data = reduce(lambda x,y : x+y,parser_all)

In [151]:
# coverting into dataframe
data_frame = pd.DataFrame(data,columns = ['filename','width','height','name','xmin','xmax','ymin','ymax'])
data_frame

,filename,width,height,name,xmin,xmax,ymin,ymax
0,000001.jpg,1024,657,car,14,301,335,522
1,000001.jpg,1024,657,car,269,571,345,489
2,000001.jpg,1024,657,car,502,798,342,450
3,000001.jpg,1024,657,car,709,1009,333,438
4,000002.jpg,800,600,car,41,768,240,497
...,...,...,...,...,...,...,...,...
7680,004954.jpg,450,500,person,104,416,30,436
7681,004955.jpg,500,374,cat,97,382,16,291
7682,004955.jpg,500,374,tvmonitor,1,441,225,373
7683,004956.jpg,333,500,tvmonitor,115,151,226,266


In [153]:
# typecasting the coloumns 
col =['width','height','xmin','ymin','xmax','ymax']
data_frame[col] = data_frame[col].astype(int)

In [155]:
data_frame['center_x'] = ((data_frame['xmax']+data_frame['xmin'])/2)/data_frame['width']
data_frame['center_y'] = ((data_frame['ymax']+data_frame['ymin'])/2)/data_frame['height']
data_frame['w'] = (data_frame['xmax']-data_frame['xmin'])/data_frame['width']
data_frame['h'] = (data_frame['ymax']-data_frame['ymin'])/data_frame['height']

In [156]:
# selecting the image name 
images = data_frame['filename'].unique()

In [158]:
# selecting tairn set and test set using train_test_split
from sklearn.model_selection import train_test_split
train_array, test_array = train_test_split(images, test_size=0.2, random_state=42)

In [159]:
train_series = pd.DataFrame(train_array,columns=['filename'])


In [161]:
test_series = pd.DataFrame(test_array,columns = ['filename'])


In [163]:
train_df = data_frame[data_frame['filename'].isin(train_series['filename'])].copy()

In [165]:
test_df = data_frame[data_frame['filename'].isin(test_series['filename'])].copy()

In [168]:
# converting images list into dataframe to select the train and test set 
# img_df = pd.DataFrame(images,columns = ['filename'])
len(test_df),len(train_df)

(1563, 6122)

In [170]:
# train_df = data_frame.query(f'filename in {train_img}')
# test_df = data_frame.query(f'filename in {test_img}')

In [171]:
# type(train_df)

In [172]:
def label_encoding(x):
    lebel = {
        'person': 0, 'car': 1, 'chair': 2, 'bottle': 3,
        'pottedplant': 4, 'bird': 5, 'dog': 6, 'sofa': 7,
        'bicycle': 8, 'horse': 9, 'boat': 10, 'motorbike': 11,
        'cat': 12, 'tvmonitor': 13, 'cow': 14, 'sheep': 15,
        'aeroplane': 16, 'train': 17, 'diningtable': 18, 'bus': 19
    }
    return lebel.get(x)

In [173]:
train_df['id'] = train_df['name'].apply(label_encoding)


In [174]:
test_df['id'] = test_df['name'].apply(label_encoding)
# train_df

In [177]:
# serating the train and test images into two different folders 
from shutil import move 

In [178]:
train_folder = 'data_images/train'
test_folder = 'data_images/test'

os.makedirs(train_folder,exist_ok=True)
# print(f"Directory '{train_folder}' created or already exists.")
os.makedirs(test_folder,exist_ok=True)
# print(f"Directory {test_folder} exist")


In [179]:
cols = ['filename' , 'id' , 'center_x',	'center_y'	,'w','h']
groupby_objects_train = train_df[cols].groupby('filename')
groupby_objects_test = test_df[cols].groupby('filename')

In [33]:
# groupby_objects_train.get_group('002064.jpg').set_index('filename').to_csv('sample.txt',index=False,header=False)
# save each  imgae in train/test foldes and respective labels in .txt


In [183]:
def save_data (filename , folder_path, group_obj):

    # move image in folders 
    src = os.path.join('data_images',filename)
    dts = os.path.join(folder_path,filename)
    move(src,dts)
    # save the label
    text_filename = os.path.join(folder_path,os.path.splitext(filename)[0] + '.txt')
    group_obj.get_group(filename).set_index('filename').to_csv(text_filename,sep=' ',index=False,header=False)


In [184]:
filename_series_train = pd.Series(groupby_objects_train.groups.keys())


In [185]:
filename_series_train.apply(save_data,args=(train_folder,groupby_objects_train))


0       None
1       None
2       None
3       None
4       None
        ... 
1979    None
1980    None
1981    None
1982    None
1983    None
Length: 1984, dtype: object

In [186]:
filename_series_test = pd.Series(groupby_objects_test.groups.keys())
filename_series_test.apply(save_data,args=(test_folder,groupby_objects_test))

0      None
1      None
2      None
3      None
4      None
       ... 
491    None
492    None
493    None
494    None
495    None
Length: 496, dtype: object